<a href="https://colab.research.google.com/github/SAfonso/paper-to-csv/blob/main/AthropicKeyTest.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install anthropic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 838.7/838.7 kB 12.2 MB/s eta 0:00:00


In [5]:
import os
import anthropic
import base64
import pandas as pd
from pathlib import Path
from google.colab import drive
from google.colab import userdata

In [6]:
drive.mount('/content/drive')

RUTA_FOTOS      = '/content/drive/MyDrive/PapelesOncle'
RUTA_SALIDA_CSV = '/content/drive/MyDrive/PapelesOncle/contactos_final.csv'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [7]:
os.environ["ANTHROPIC_API_KEY"] = userdata.get('ANTHROPIC_API_KEY')

In [8]:
def extraer_datos_con_claude(ruta_imagen):
    client = anthropic.Anthropic()

    ext = Path(ruta_imagen).suffix.lower()
    media_type = "image/jpeg" if ext in [".jpg", ".jpeg"] else "image/png"

    with open(ruta_imagen, "rb") as f:
        img_b64 = base64.standard_b64encode(f.read()).decode("utf-8")

    mensaje = client.messages.create(
        model="claude-opus-4-6",
        max_tokens=256,
        messages=[{
            "role": "user",
            "content": [
                {
                    "type": "image",
                    "source": {
                        "type": "base64",
                        "media_type": media_type,
                        "data": img_b64,
                    },
                },
                {
                    "type": "text",
                    "text": (
                        "Esta es una tarjeta de sorteo de Medianamente Comedy "
                        "con dos campos manuscritos: NOMBRE y Email.\n"
                        "Extrae exactamente el nombre y el email escritos a mano.\n"
                        "Responde ÚNICAMENTE en este formato, sin nada más:\n"
                        "NOMBRE: <nombre>\n"
                        "EMAIL: <email>\n\n"
                        "Si no puedes leer algún campo, escribe ILEGIBLE."
                    )
                }
            ],
        }],
    )

    respuesta = mensaje.content[0].text.strip()
    nombre, email = "No detectado", "No detectado"

    for linea in respuesta.split("\n"):
        if linea.startswith("NOMBRE:"):
            nombre = linea.replace("NOMBRE:", "").strip()
        elif linea.startswith("EMAIL:"):
            email = linea.replace("EMAIL:", "").strip()

    return nombre, email

In [ ]:
lista_contactos = []

archivos = sorted([
    f for f in os.listdir(RUTA_FOTOS)
    if f.lower().endswith(('.jpg', '.jpeg', '.png'))
])

print(f"Imágenes encontradas: {len(archivos)}\n")

for archivo in archivos:
    ruta = os.path.join(RUTA_FOTOS, archivo)
    try:
        nombre, email = extraer_datos_con_claude(ruta)
        print(f"✅ [{archivo}] -> {nombre} | {email}")
        lista_contactos.append({"archivo": archivo, "nombre": nombre, "email": "email"})
    except Exception as e:
        print(f"❌ [{archivo}] -> ERROR: {e}")
        lista_contactos.append({"archivo": archivo, "nombre": "ERROR", "email": str(e)})
